# Tekton 파이프라인용 MLflow 웹훅 설정하기

이 노트북에서는 프롬프트 관련 이벤트가 발생할 때 Tekton 파이프라인을 자동으로 트리거하는 MLflow 웹훅을 만듭니다.

연결할 파이프라인은 두 개입니다.

1. **평가 파이프라인**: 새 프롬프트 버전이 생성되면 트리거되어, 해당 버전에 대한 평가를 자동으로 실행합니다.
2. **프롬프트 승격(Prompt Promotion) 파이프라인**: 누군가 프롬프트 버전에 `prod` 별칭을 태그하면 트리거되어, 프로덕션 GitOps 설정을 자동으로 업데이트합니다.

In [ ]:
# ‼️‼️ 실제 값으로 바꾸세요 ‼️
USER_NAME = "<your-user-name>"
CLUSTER_DOMAIN = "<your-cluster-domain>"

In [ ]:
import mlflow

MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_workspace(f"{USER_NAME}-toolings")

client = mlflow.MlflowClient()

## 1. 평가 파이프라인용 웹훅

이 웹훅은 새 프롬프트나 프롬프트 버전이 생성될 때마다 **canopy-evals** Tekton EventListener에 알리도록 MLflow를 설정합니다. 이렇게 하면 프롬프트가 변경될 때마다 평가 파이프라인이 자동으로 트리거됩니다.

In [ ]:
webhook = client.create_webhook(
    name=f"{USER_NAME}-canopy-evals",
    url=f"https://canopy-evals-event-listener-{USER_NAME}-toolings.{CLUSTER_DOMAIN}",
    events=["prompt.created", "prompt_version.created"],
    description="Triggers evaluation pipeline when new prompt versions are created",
)

print(f"Created webhook: {webhook.webhook_id}")

### ❣️ 이제 실습 안내 문서로 돌아가 방금 설정한 자동화를 테스트해 보세요. 더 많은 자동화를 하러 다시 돌아올 거예요, 약속해요 😏

## 2. 프롬프트 승격 파이프라인용 웹훅

이 웹훅은 프롬프트 버전에 `prod` 별칭이 추가되면 실행됩니다. **prompt-promotion** 파이프라인을 트리거하여 프로덕션 GitOps 설정(`genaiops-gitops/canopy/prod/backend/config.yaml`)을 특정 프롬프트 버전 번호로 업데이트하므로, 프로덕션에서 무엇이 실행 중인지 완전히 추적할 수 있습니다.

> **참고:** ArgoCD로 `prompt-promotion-pipeline` 차트를 배포한 후에만 이 셀을 실행하세요.

In [ ]:
webhook = client.create_webhook(
    name=f"{USER_NAME}-canopy-prompt-promotion",
    url=f"https://prompt-promotion-event-listener-{USER_NAME}-toolings.{CLUSTER_DOMAIN}",
    events=["prompt_alias.created"],
    description="Triggers prompt promotion pipeline when prod alias is set",
)

print(f"Created webhook: {webhook.webhook_id}")